# Labeling

## Housekeeping (no interaction required)

In [1]:
%pip install openrouter
# Is dotenv installed?

/home/bode-wsl/projects/summerschool26/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import random
import time
from pathlib import Path
import json
import pandas as pd
from tqdm.notebook import tqdm

tqdm.pandas()

In [2]:
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
DATA_DIR = Path('/content/drive/MyDrive/ZBSummerSchool26/data') if IN_COLAB else Path('../data')
MODEL_NAME = "openai/gpt-5.4-mini"

In [3]:
def confirm(question: str = "Do you want to execute this cell?"):
    while True:
        response = input(f"{question} (y/n): ").lower()
        if response in ["y", "yes"]:
            return True
        elif response in ["n", "no"]:
            return False
        else:
            print("Please enter 'y' or 'n'.")

## Setup (Interaction required)

### Load the data

In [6]:
### ⬇️⬇️⬇️ 💽 Adjust here if you want to continue with your own query
CORPUS_NAME = "armensteuer_and_similars"
LOAD_OWN_DATA = False
### ⬆️⬆️⬆️

In [ ]:
if IN_COLAB and LOAD_OWN_DATA: # and confirm("Do you want to mount your Google Drive?"):
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DATA_DIR, exist_ok=True)

#### <img src="https://cdn.simpleicons.org/googledrive" alt="💾" width=16> Load your own data from Google Drive

#### <img src="https://cdn.svglogos.dev/logos/google-drive.svg" alt="💾" width=16> Load your own data from Google Drive

In [ ]:
if LOAD_OWN_DATA:
    RAWDATA_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.parquet" # Use data from filtering module
    raw_df = pd.read_parquet(RAWDATA_PATH)

    SENTENCES_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.sentences.parquet" # Use data from filtering module
    sentences_df = pd.read_parquet(SENTENCES_PATH)

    raw_df = raw_df.join(sentences_df, on="id")

In [ ]:

if not LOAD_OWN_DATA:
    # RAWDATA_ORIGIN_URL = "https://github.com/pleyad/Summer-School-2026/raw/refs/heads/main/data/armensteuer_and_similars.filtered.csv" 
    # raw_df = pd.read_csv(RAWDATA_ORIGIN_URL, index_col="id")

    RAWDATA_ORIGIN_URL = "https://github.com/pleyad/Summer-School-2026/raw/refs/heads/main/data/armensteuer_and_similars.filtered.parquet"
    print(f"Loading raw data ...", end="\r")
    raw_df = pd.read_parquet(RAWDATA_ORIGIN_URL)

    SENTENCES_URL = "https://github.com/pleyad/Summer-School-2026/raw/refs/heads/main/data/armensteuer_and_similars.filtered.sentences.parquet"
    print(f"Loading sentences data ...", end="\r")
    sentences_df = pd.read_parquet(SENTENCES_URL)

    raw_df = raw_df.join(sentences_df)

### Enable LLM interaction

In [9]:
from openrouter import OpenRouter
import os
import dotenv
dotenv.load_dotenv()

with OpenRouter(
  # api_key=API_KEY,
  api_key=os.getenv("OPENROUTER_API_KEY"),
) as client:
  response = client.chat.send(
    model=MODEL_NAME,
    messages=[
      {
        "role": "user",
        "content": "What is the meaning of life?"
      }
    ]
  )

print(response.choices[0].message.content)

There isn’t one single universally agreed meaning of life.

Different perspectives give different answers:

- **Philosophy:** meaning is something we create through values, relationships, and chosen goals.
- **Religion/spirituality:** meaning comes from connection to God, the universe, or a higher purpose.
- **Psychology:** meaning often comes from belonging, growth, love, and contributing to something beyond yourself.
- **A practical view:** life’s meaning may be to experience, learn, help others, and build a life you find worthwhile.

A concise answer: **the meaning of life is what you make of it**—often shaped by love, purpose, and connection.

If you want, I can also give you:
1. a philosophical answer,  
2. a religious answer, or  
3. a funny answer.


### 

### Parsing

In [10]:
raw_df["year"] = pd.to_datetime(raw_df["date"], errors="coerce").dt.year

## Introduction to Prompting

## Hands-on:
Now try the following things:

1.   Change the user prompt and re-run the request.
2.   Change the system prompt to something completely random and re-run the request.

### Temperature and top-p
Let's go through each of these arguments one-by-one. Below you will see API calling code for the LLM with a slightly more complex prompt. Try out requests to this LLM where you:


1.   Vary the temperature parameter.
2. Vary the top-p parameter.

Your task is try to get the funniest and weirdest response possible from the model.

In order to prevent too excessive an output (and to protect our OpenAI budget!) we'll limit the output to a maximum of 50 tokens using the `max_tokens` argument; please do not change this.

In [10]:
# ⬇️⬇️⬇️ Adjust the system and user prompts here to see how it affects the output
longer_system_prompt = (
    "You are a political text classifier. Given a social media post, return a JSON object "
    "with exactly four fields: "
    "\"topic\" (a short label for the main topic, e.g. 'economy', 'healthcare', 'immigration'), "
    "\"sentiment\" (one of: 'positive', 'negative', 'neutral'), and "
    "\"is_political\" (true or false). "
    "Return only the JSON object, nothing else."
)
longer_user_prompt = (
    "Post: 'Can't believe they're cutting the healthcare budget again while giving tax breaks "
    "to billionaires. This government doesn't care about ordinary people.'"
)
# ⬆️⬆️⬆️

In [11]:
with OpenRouter(
  api_key=os.getenv("OPENROUTER_API_KEY"),
) as client:
    response = client.chat.send(
        model=MODEL_NAME,
        messages=[
        {"role": "system", "content": longer_system_prompt},
        {"role": "user", "content": longer_user_prompt}
        ],
        max_tokens=50,

        # ⬇️⬇️⬇️ Adjust these parameters to see how it affects the output
        temperature=1, # 0-1
        top_p=0.1 # 0-1
        # ⬆️⬆️⬆️
    )

output_text = response.choices[0].message.content  # take the output and extract the text response only

# parse and pretty-print the JSON output
parsed = json.loads(output_text)
print(json.dumps(parsed, indent=2))

{
  "topic": "healthcare",
  "sentiment": "negative",
  "is_political": true
}


## Parse the data from LabelStudio

In [ ]:
if LOAD_OWN_DATA:
    LABELLED_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.labelled.csv" # Use data from filtering module
    labelled_df = pd.read_csv(LABELLED_PATH, index_col="id")


if not LOAD_OWN_DATA:
    RAWDATA_ORIGIN_URL = "https://github.com/pleyad/Summer-School-2026/raw/refs/heads/main/data/armensteuer_and_similars.filtered.parquet"
    print(f"Loading raw data ...", end="\r")
    labelled_df = pd.read_parquet(RAWDATA_ORIGIN_URL)